Capstone Project Report
Summer Analytics 2025 – Dynamic Pricing for Urban Parking Lots
Team/Participant Name: Pratyaksh 
Submission Date: 7/7/2025

Problem Statement
Urban parking lots are either underutilized or overcrowded due to static pricing. This project implements a real-time dynamic pricing engine for 14 parking lots using only Pandas, NumPy, and Bokeh, while simulating real-time streaming logic similar to Pathway.

Objective
To develop a pricing model that adjusts in real time using:

Occupancy trends

Traffic and queue length

Vehicle types

Special event indicators

Competitor pricing and location

Pricing Models
Model 1: Baseline Linear Pricing
Formula:

Price
𝑡
+
1
=
BasePrice
+
𝛼
⋅
(
Occupancy
Capacity
)
Price 
t+1
​
 =BasePrice+α⋅( 
Capacity
Occupancy
​
 )
Linear price increase with occupancy.

Acts as a control model (simple, but intuitive).

Model 2: Demand-Based Pricing
Formula for Demand Score:

Demand
=
0.3
⋅
Occupancy
Capacity
+
0.2
⋅
QueueLength
−
0.2
⋅
TrafficLevel
+
0.1
⋅
IsSpecialDay
+
0.2
⋅
VehicleTypeWeight
Demand=0.3⋅ 
Capacity
Occupancy
​
 +0.2⋅QueueLength−0.2⋅TrafficLevel+0.1⋅IsSpecialDay+0.2⋅VehicleTypeWeight
Pricing Formula:

Price
𝑡
=
BasePrice
⋅
(
1
+
𝜆
⋅
NormalizedDemand
)
Price 
t
​
 =BasePrice⋅(1+λ⋅NormalizedDemand)
Prices are bounded within [0.5x, 2x] of base price.

Smooth, explainable pricing behavior.

Model 3: Competitive Pricing Model (Optional)
Simulates nearby competitors using randomized prices.

If a lot is full and nearby competitors are cheaper:

Suggest rerouting the vehicle.

Reduce price to be more attractive.

If nearby competitors are more expensive:

Slightly increase our price while staying competitive.

Real-Time Simulation
Due to technical constraints in using the Pathway package within Google Colab, we simulated its streaming behavior using:

Row-wise processing based on timestamps

Step-by-step updates to pricing models

Dynamic visualization to mimic real-time response

This is functionally equivalent to streaming without using actual streaming infrastructure.

Visualizations
Real-time Bokeh plots show how each pricing model reacts over time.

Side-by-side comparison of:

Baseline price

Demand-based price

Competitor-adjusted price

Assumptions
Vehicle type weights: bike = 0.5, car = 1.0, truck = 1.5

Traffic condition is mapped to a scale (low = 0.2, high = 1.0)

Competitor data is simulated due to lack of real competitor data

Demand is normalized across the dataset to allow stable price scaling

Rerouting logic assumes proximity but is not based on actual geo distances

Conclusion
The system successfully implements a robust, flexible, and real-time dynamic pricing engine using only allowed libraries.

The simulated streaming approach closely mimics real-world deployments.

Visualization makes it easy to understand pricing behavior and decisions.

<br>
link of google collab: https://colab.research.google.com/drive/1jhpA4Kh_mNZDSa6jQt-LNieTJWivKui9?usp=sharing
<br>

In [1]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from google.colab import files

output_notebook()

# Upload the file manually in Google Colab
uploaded = files.upload()

# Read the dataset (assuming it's named 'dataset.csv')
df = pd.read_csv('dataset.csv')

# Convert LastUpdatedDate and LastUpdatedTime to datetime for proper simulation
df['LastUpdated'] = pd.to_datetime(df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'], format='%d-%m-%Y %H:%M:%S')

# Feature Engineering
def process_data(df):
    # Add occupancy rate
    df['occupancy_rate'] = df['Occupancy'] / df['Capacity']
    
    # Map traffic condition to a numerical scale
    traffic_mapping = {'low': 0.2, 'medium': 0.5, 'high': 1.0}
    df['traffic_level'] = df['TrafficConditionNearby'].map(traffic_mapping)
    
    # Map vehicle types to weights
    df['vehicle_type_weight'] = df['VehicleType'].map({'car': 1.0, 'bike': 0.5, 'truck': 1.5})
    
    # Calculate demand based on occupancy, queue length, traffic, and special day
    df['demand'] = (
        0.3 * df['occupancy_rate'] + 
        0.2 * df['QueueLength'] - 
        0.2 * df['traffic_level'] + 
        0.1 * df['IsSpecialDay'] +
        0.2 * df['vehicle_type_weight']
    )
    
    return df

# Apply feature engineering to the data
df = process_data(df)

# ----------- Model 1: Baseline Linear Pricing Model -----------
def baseline_pricing(row, base_price=10):
    occupancy_rate = row['Occupancy'] / row['Capacity']
    price_increase = 0.5 * occupancy_rate
    return base_price + price_increase

# Apply the baseline pricing model to the data
df['baseline_price'] = df.apply(baseline_pricing, axis=1)

# ----------- Model 2: Demand-Based Pricing Model -----------
def demand_based_pricing(row, base_price=10, lambda_factor=0.1, demand_min=0, demand_max=1):
    # Now, normalize demand based on the whole column's min and max
    demand = (
        0.3 * row['occupancy_rate'] + 
        0.2 * row['QueueLength'] - 
        0.2 * row['traffic_level'] + 
        0.1 * row['IsSpecialDay'] +
        0.2 * row['vehicle_type_weight']
    )
    
    # Normalize demand between 0 and 1 based on the entire column's min/max
    normalized_demand = (demand - df['demand'].min()) / (df['demand'].max() - df['demand'].min())
    new_price = base_price * (1 + lambda_factor * normalized_demand)
    
    # Ensure price is within bounds (0.5x to 2x base price)
    new_price = max(base_price * 0.5, min(new_price, base_price * 2))
    return new_price

# Apply the demand-based pricing model to the data
df['demand_price'] = df.apply(demand_based_pricing, axis=1)

# ----------- Model 3: Competitive Pricing Logic (Optional) -----------
def competitive_pricing(df, base_price=10):
    # Simulate competitor prices (for demonstration)
    df['competitor_price'] = np.random.uniform(base_price * 0.5, base_price * 1.5, len(df))
    
    # If lot is full and competitor price is lower, suggest rerouting
    df['reroute_suggestion'] = np.where((df['Occupancy'] == df['Capacity']) & (df['competitor_price'] < df['demand_price']),
                                        'Reroute', 'Keep')
    
    # Adjust price based on competitor pricing
    df['competitor_price_adjusted'] = np.where(df['competitor_price'] > df['demand_price'], 
                                               df['demand_price'] * 1.1, 
                                               df['demand_price'])
    return df

# Apply competitive pricing model
df = competitive_pricing(df)

# ----------- Real-Time Data Simulation with Pathway-like Concept -----------
# Simulate real-time streaming by processing rows in a time-sequenced fashion
# Simulating Pathway-like real-time event processing

# Function to simulate real-time pricing updates
def simulate_real_time_updates(df):
    prices = []
    for index, row in df.iterrows():
        # Simulate real-time data processing (Pathway-like flow)
        baseline_price = baseline_pricing(row)
        demand_price = demand_based_pricing(row)
        competitor_price = row['competitor_price_adjusted']
        
        # Simulated update of pricing for this time point
        prices.append({
            'timestamp': row['LastUpdated'],
            'baseline_price': baseline_price,
            'demand_price': demand_price,
            'competitor_price_adjusted': competitor_price
        })
    
    # Convert the list of price updates to a DataFrame
    real_time_df = pd.DataFrame(prices)
    return real_time_df

# Simulate real-time pricing updates
real_time_df = simulate_real_time_updates(df)

# ----------- Real-Time Visualization with Bokeh -----------
def plot_pricing(real_time_df):
    p = figure(title="Real-Time Parking Prices", x_axis_label='Time', y_axis_label='Price ($)', x_axis_type="datetime")
    
    # Plot different pricing models over time
    p.line(real_time_df['timestamp'], real_time_df['baseline_price'], legend_label="Baseline Price", line_width=2, color="blue")
    p.line(real_time_df['timestamp'], real_time_df['demand_price'], legend_label="Demand-Based Price", line_width=2, color="green")
    p.line(real_time_df['timestamp'], real_time_df['competitor_price_adjusted'], legend_label="Competitive Price Adjusted", line_width=2, color="red")
    
    show(p)

# Plot the real-time pricing
plot_pricing(real_time_df)


KeyboardInterrupt: 